In [0]:
# =============================================================================
# NB15 — Band gap model: revalidation under paper-, composition- and
#        family-grouped CV
# =============================================================================
# Context: decision (2026-10-14) — the article is framed as a critical
# benchmark. Its positive part is that an INTRINSIC property (band gap) is
# learnable from composition. The earlier R² ≈ 0.70 (progress report) used a
# compound-grouped split on an older dataset; it must pass the same rigour as
# the activity analyses before it is reported.
#
# What this notebook does
#   1. Target: measured band gap (master.bandgap_eV). One row per
#      (doi_key, Compound_original) — repeated activity runs of the same sample
#      would otherwise be counted several times.
#   2. Features: X_material compositional/structural descriptors.
#      LEAKAGE GUARD: ECB_* / EVB_* columns are EXCLUDED — NB03 computed them
#      from the measured band gap itself (ECB = χ − 4.5 − Eg/2).
#   3. Three grouping schemes, 5×5 repeated grouped CV each:
#        DOI         — unseen paper (same composition may appear elsewhere)
#        composition — unseen base_compound
#        family      — unseen family_proposed (strictest extrapolation)
#   4. Baselines: train mean; family mean (train-fold mean of the same family);
#      single-feature linear model on wav_gs_bandgap (elemental DFT gap).
#   5. Models: Ridge, Random Forest, XGBoost. Pooled out-of-fold R², MAE, RMSE.
#   6. Uncertainty that INCLUDES sampling variance: cluster bootstrap over
#      groups of the repeat-0 OOF predictions (1000 resamples).
#   7. y-randomization: band gap permuted across rows (N_YRAND fits).
#
# Note: descriptors are computed per host (base_compound), so doped variants
# of one host share features but differ in measured Eg. That within-host
# spread is an irreducible error for this feature set and is reported.
#
# Inputs:  ml_perovskites.master, ml_perovskites.X_material
# Outputs: ml_perovskites.bandgap_cv_nb15, ml_perovskites.bandgap_oof_nb15,
#          /tmp/nb15_parity.png
# Rules:   imputation/scaling inside folds; groups never straddle train/val;
#          SEED = 42
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import re, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.impute        import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline      import make_pipeline
from sklearn.linear_model  import RidgeCV, LinearRegression
from sklearn.ensemble      import RandomForestRegressor
from sklearn.metrics       import r2_score, mean_absolute_error, mean_squared_error
from pyspark.sql import SparkSession

SEED, SCHEMA, N_FOLDS, N_REPEATS = 42, "ml_perovskites", 5, 5
N_BOOT, N_YRAND = 1000, 20
EG_RANGE = (0.0, 6.5)   # eV; values outside are flagged and excluded

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "xgboost"])
    import xgboost as xgb

# ---------------------------------------------------------------------------
# 1. Load, deduplicate, define features
# ---------------------------------------------------------------------------
master    = spark.table(f"{SCHEMA}.master").toPandas()
X_mat_raw = spark.table(f"{SCHEMA}.X_material").toPandas()
for t in [master, X_mat_raw]:
    t["source_row_id"] = t["source_row_id"].astype(str).str.strip()
for c in X_mat_raw.columns:
    if c != "source_row_id":
        X_mat_raw[c] = pd.to_numeric(X_mat_raw[c], errors="coerce")

master["eg"] = pd.to_numeric(master["bandgap_eV"], errors="coerce")
d = master[master["eg"].notna() & master["doi_key"].notna() &
           master["base_compound"].notna()].copy()
bad = d[(d["eg"] < EG_RANGE[0]) | (d["eg"] > EG_RANGE[1])]
print(f"Rows with measured Eg: {len(d)} | outside {EG_RANGE} eV (flagged, excluded): "
      f"{len(bad)} {bad['source_row_id'].head(10).tolist()}")
d = d[(d["eg"] >= EG_RANGE[0]) & (d["eg"] <= EG_RANGE[1])]

sample_key = "Compound_original" if "Compound_original" in d.columns else "base_compound"
d["family"] = (d["family_proposed"].astype(str).str.strip()
               if "family_proposed" in d.columns else d["base_compound"])
d = (d.groupby(["doi_key", sample_key], as_index=False)
       .agg(source_row_id=("source_row_id", "first"), eg=("eg", "mean"),
            eg_sd=("eg", "std"), base_compound=("base_compound", "first"),
            family=("family", "first")))
print(f"Samples after dedup (doi_key × {sample_key}): {len(d)} | DOIs "
      f"{d['doi_key'].nunique()} | compositions {d['base_compound'].nunique()} | "
      f"families {d['family'].nunique()}")
print(f"  Eg range {d['eg'].min():.2f}–{d['eg'].max():.2f} eV, mean {d['eg'].mean():.2f}, "
      f"SD {d['eg'].std():.2f}")

META = {"source_row_id", "base_compound", "family_proposed", "formula_base_paper",
        "A_site_paper", "B_site_paper", "doi", "doi_key", "y_log10_1p_final",
        "y_log10_1p", "h2_rate_umol_g_h", "h2_umol_h_rate_mass"}
LEAK = [c for c in X_mat_raw.columns if re.match(r"^(ECB|EVB)_", c)]
feats = [c for c in X_mat_raw.columns if c not in META and c not in LEAK
         and pd.api.types.is_numeric_dtype(X_mat_raw[c])
         and X_mat_raw[c].isna().mean() < 0.50]
print(f"Features: {len(feats)} | EXCLUDED as target-derived: {LEAK}")

d = d.merge(X_mat_raw[["source_row_id"] + feats], on="source_row_id", how="left")

# Irreducible spread: same host, same paper, different measured Eg
host_sd = d.groupby(["doi_key", "base_compound"])["eg"].std().dropna()
print(f"Within-host Eg spread (same paper, ≥2 samples of one host): "
      f"{len(host_sd)} groups, median SD {host_sd.median():.3f} eV — "
      f"descriptors cannot resolve this")

SCHEMES = {"DOI": "doi_key", "composition": "base_compound", "family": "family"}

# ---------------------------------------------------------------------------
# 2. Models and baselines
# ---------------------------------------------------------------------------
def make_models():
    # Imputation is done per fold in oof(); XGBoost is kept OUT of sklearn
    # Pipeline on purpose (known xgboost / scikit-learn>=1.6 tag incompatibility).
    return {
        "Ridge":   make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 4, 25))),
        "RF":      RandomForestRegressor(n_estimators=500, min_samples_leaf=2,
                                         max_features=0.3, n_jobs=-1, random_state=SEED),
        "XGBoost": xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                                    subsample=0.8, colsample_bytree=0.6, min_child_weight=2,
                                    reg_lambda=1.0, tree_method="hist",
                                    random_state=SEED, verbosity=0),
    }

def folds(groups, rep):
    g = np.array(sorted(pd.unique(groups)))
    k = min(N_FOLDS, len(g))
    perm = np.random.default_rng(SEED + rep).permutation(g)
    f_of = {x: i % k for i, x in enumerate(perm)}
    return np.array([f_of[x] for x in groups]), k

def oof(groups, rep, yvals, model_names=None):
    f, k = folds(groups, rep)
    X = d[feats].values.astype(float)
    out = {}
    names = model_names or ["mean", "family_mean", "gs_bandgap_linear",
                            "Ridge", "RF", "XGBoost"]
    for n in names:
        out[n] = np.full(len(d), np.nan)
    for fold in range(k):
        tr, va = np.where(f != fold)[0], np.where(f == fold)[0]
        assert not set(groups[tr]) & set(groups[va]), "group leakage!"
        ytr = yvals[tr]
        if "mean" in names:
            out["mean"][va] = ytr.mean()
        if "family_mean" in names:
            fam_mean = pd.Series(ytr).groupby(d["family"].values[tr]).mean()
            out["family_mean"][va] = d["family"].iloc[va].map(fam_mean).fillna(ytr.mean()).values
        imp = SimpleImputer(strategy="median")
        Xtr, Xva = imp.fit_transform(X[tr]), imp.transform(X[va])
        kept = [c for c, s in zip(feats, imp.statistics_) if not np.isnan(s)]
        if "gs_bandgap_linear" in names and "wav_gs_bandgap" in kept:
            j = kept.index("wav_gs_bandgap")
            m = LinearRegression().fit(Xtr[:, [j]], ytr)
            out["gs_bandgap_linear"][va] = m.predict(Xva[:, [j]])
        for n, m in make_models().items():
            if n in names:
                m.fit(Xtr, ytr)
                out[n][va] = m.predict(Xva)
    return out

def metrics(y, p):
    ok = ~np.isnan(p)
    return (r2_score(y[ok], p[ok]), mean_absolute_error(y[ok], p[ok]),
            float(np.sqrt(mean_squared_error(y[ok], p[ok]))))

# ---------------------------------------------------------------------------
# 3. Repeated grouped CV for each scheme
# ---------------------------------------------------------------------------
y = d["eg"].values
rows, oof_store = [], []
for scheme, gcol in SCHEMES.items():
    groups = d[gcol].astype(str).values
    for rep in range(N_REPEATS):
        preds = oof(groups, rep, y)
        for n, p in preds.items():
            if np.isnan(p).all():
                continue
            r2, mae, rmse = metrics(y, p)
            rows.append({"scheme": scheme, "model": n, "repeat": rep,
                         "r2": r2, "mae_eV": mae, "rmse_eV": rmse})
            if rep == 0:
                oof_store.append(pd.DataFrame({"scheme": scheme, "model": n,
                                               "source_row_id": d["source_row_id"],
                                               "group": groups, "eg": y, "pred": p}))
    print(f"  scheme {scheme} done ({len(pd.unique(groups))} groups)")

res = pd.DataFrame(rows)
oof_df = pd.concat(oof_store, ignore_index=True)

summary = (res.groupby(["scheme", "model"])
              .agg(r2=("r2", "mean"), r2_min=("r2", "min"), r2_max=("r2", "max"),
                   mae_eV=("mae_eV", "mean"), rmse_eV=("rmse_eV", "mean"))
              .reset_index())

# ---------------------------------------------------------------------------
# 4. Cluster bootstrap CI (repeat-0 OOF; resampling groups of the scheme)
# ---------------------------------------------------------------------------
rng = np.random.default_rng(SEED + 7)
ci = []
for (scheme, model), g in oof_df.groupby(["scheme", "model"]):
    g = g.dropna(subset=["pred"])
    by = {k: v.index.values for k, v in g.groupby("group")}
    keys = np.array(list(by.keys()))
    vals = []
    for _ in range(N_BOOT):
        idx = np.concatenate([by[k] for k in rng.choice(keys, len(keys), replace=True)])
        yy, pp = g.loc[idx, "eg"].values, g.loc[idx, "pred"].values
        if np.var(yy) > 0:
            vals.append(r2_score(yy, pp))
    ci.append({"scheme": scheme, "model": model,
               "r2_ci_lo": np.percentile(vals, 2.5), "r2_ci_hi": np.percentile(vals, 97.5)})
summary = summary.merge(pd.DataFrame(ci), on=["scheme", "model"], how="left")

pd.set_option("display.width", 200)
order = {"mean": 0, "family_mean": 1, "gs_bandgap_linear": 2, "Ridge": 3, "RF": 4, "XGBoost": 5}
summary["o"] = summary["model"].map(order)
summary = summary.sort_values(["scheme", "o"]).drop(columns="o")
print(f"\n--- Band gap, pooled OOF metrics ({N_REPEATS} repeats; r2_min/max over repeats; "
      f"CI = cluster bootstrap over groups, repeat 0) ---")
print(summary.round(3).to_string(index=False))

# ---------------------------------------------------------------------------
# 5. y-randomization (DOI scheme, XGBoost)
# ---------------------------------------------------------------------------
g_doi = d["doi_key"].astype(str).values
r2_rand = []
rng_y = np.random.default_rng(SEED + 99)
for _ in range(N_YRAND):
    yr = rng_y.permutation(y)
    p = oof(g_doi, 0, yr, model_names=["XGBoost"])["XGBoost"]
    r2_rand.append(r2_score(yr, p))
obs = summary[(summary["scheme"] == "DOI") & (summary["model"] == "XGBoost")]["r2"].iloc[0]
print(f"\ny-randomization (XGBoost, DOI scheme, {N_YRAND} shuffles): "
      f"R² mean {np.mean(r2_rand):+.3f}, max {np.max(r2_rand):+.3f} | observed {obs:.3f}")

print("\nReading guide:")
print("  * Report the composition- and family-grouped numbers, not only DOI-grouped.")
print("  * A model is useful only if it beats family_mean in the same scheme.")
print("  * The bootstrap CI includes sampling variance; the r2_min/max range does not.")

# ---------------------------------------------------------------------------
# 6. Parity plots (XGBoost, repeat 0) + write
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 4.3))
lim = (max(0, y.min() - 0.2), y.max() + 0.2)
for ax, scheme in zip(axes, SCHEMES):
    g = oof_df[(oof_df["scheme"] == scheme) & (oof_df["model"] == "XGBoost")]
    s = summary[(summary["scheme"] == scheme) & (summary["model"] == "XGBoost")].iloc[0]
    ax.scatter(g["eg"], g["pred"], s=12, alpha=0.6)
    ax.plot(lim, lim, color="black", lw=0.8)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("Measured Eg (eV)"); ax.set_ylabel("Predicted Eg (eV)")
    ax.set_title(f"{scheme}-grouped: R² {s['r2']:.2f} "
                 f"[{s['r2_ci_lo']:.2f}, {s['r2_ci_hi']:.2f}], MAE {s['mae_eV']:.2f} eV",
                 fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/nb15_parity.png", dpi=130); plt.close()
print("\nSaved: /tmp/nb15_parity.png")

spark.createDataFrame(res).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.bandgap_cv_nb15")
spark.createDataFrame(oof_df.replace({np.nan: None})).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.bandgap_oof_nb15")
print(f"✓ Written: {SCHEMA}.bandgap_cv_nb15 ({len(res)} rows), "
      f"{SCHEMA}.bandgap_oof_nb15 ({len(oof_df)} rows)")
print("\n=== NB15 COMPLETE ===")

In [0]:
# =============================================================================
# NB15b — Paper effect on the measured band gap
# =============================================================================
# Context (NB15): composition-grouped CV gave R² ≈ 0.43, but paper-grouped CV
# only ≈ 0.13–0.19 (bootstrap CI ~[0.00, 0.44]). A higher score when OTHER
# samples of the same paper stay in training indicates a paper-level offset in
# the reported Eg (Tauc method, direct vs indirect assumption, instrument,
# extrapolation choices). This notebook quantifies it.
#
#   1. Inter-paper reproducibility: for compositions whose Eg is reported by
#      ≥ 2 papers, the spread of Eg across papers for the SAME composition
#      (a measurement-noise floor that no composition-only model can beat).
#   2. ICC(1) of Eg by DOI (one-way random effects) on deduplicated samples:
#      share of Eg variance shared within a paper.
#   3. Strictest feasible validation: DOI-grouped CV with COMPOSITION PURGING —
#      for each validation fold, every training sample whose base_compound
#      also appears in the validation fold is removed. No paper and no
#      composition is shared between train and validation.
#   4. Copied-value check: share of multi-sample hosts (same paper) whose
#      samples all carry an identical Eg (possible host value copied to
#      variants during curation).
#
# Inputs:  ml_perovskites.master, ml_perovskites.X_material
# Outputs: ml_perovskites.bandgap_purged_nb15b, /tmp/nb15b_reproducibility.png
# Rules:   same dedup and leakage guard as NB15 (ECB_*/EVB_* excluded); SEED=42
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import re, subprocess, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.impute       import SimpleImputer
from sklearn.ensemble     import RandomForestRegressor
from sklearn.metrics      import r2_score, mean_absolute_error
from pyspark.sql import SparkSession

SEED, SCHEMA, N_FOLDS, N_REPEATS, N_BOOT = 42, "ml_perovskites", 5, 5, 1000

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "xgboost"])
    import xgboost as xgb

# ---------------------------------------------------------------------------
# 1. Load + dedup (identical to NB15)
# ---------------------------------------------------------------------------
master    = spark.table(f"{SCHEMA}.master").toPandas()
X_mat_raw = spark.table(f"{SCHEMA}.X_material").toPandas()
for t in [master, X_mat_raw]:
    t["source_row_id"] = t["source_row_id"].astype(str).str.strip()
for c in X_mat_raw.columns:
    if c != "source_row_id":
        X_mat_raw[c] = pd.to_numeric(X_mat_raw[c], errors="coerce")

master["eg"] = pd.to_numeric(master["bandgap_eV"], errors="coerce")
d = master[master["eg"].between(0.0, 6.5) & master["doi_key"].notna() &
           master["base_compound"].notna()].copy()
sample_key = "Compound_original" if "Compound_original" in d.columns else "base_compound"
d = (d.groupby(["doi_key", sample_key], as_index=False)
       .agg(source_row_id=("source_row_id", "first"), eg=("eg", "mean"),
            base_compound=("base_compound", "first")))
d["base_compound"] = d["base_compound"].astype(str).str.strip()
print(f"Samples: {len(d)} | DOIs {d['doi_key'].nunique()} | compositions "
      f"{d['base_compound'].nunique()}")

# ---------------------------------------------------------------------------
# 2. Inter-paper reproducibility of Eg for the same composition
# ---------------------------------------------------------------------------
per_paper = d.groupby(["base_compound", "doi_key"])["eg"].mean().reset_index()
multi = per_paper.groupby("base_compound").filter(lambda g: len(g) >= 2)
rep = (multi.groupby("base_compound")["eg"]
            .agg(n_papers="size", mean="mean", sd="std",
                 range=lambda s: s.max() - s.min())
            .sort_values("n_papers", ascending=False))
print(f"\n--- Inter-paper reproducibility (compositions reported by ≥2 papers: {len(rep)}) ---")
if len(rep):
    print(f"  Eg SD across papers, same composition: median {rep['sd'].median():.3f} eV, "
          f"mean {rep['sd'].mean():.3f} eV | range median {rep['range'].median():.2f} eV, "
          f"max {rep['range'].max():.2f} eV")
    print(rep.head(12).round(3).to_string())

# ---------------------------------------------------------------------------
# 3. ICC(1) of Eg by DOI (one-way random effects, unbalanced groups)
# ---------------------------------------------------------------------------
def icc1(values, groups):
    df_ = pd.DataFrame({"v": values, "g": groups})
    k = df_.groupby("g").size()
    df_ = df_[df_["g"].isin(k[k >= 1].index)]
    a, n = df_["g"].nunique(), len(df_)
    gm = df_["v"].mean()
    msb = (df_.groupby("g")["v"].agg(lambda s: len(s) * (s.mean() - gm) ** 2).sum()) / (a - 1)
    msw = (df_.groupby("g")["v"].agg(lambda s: ((s - s.mean()) ** 2).sum()).sum()) / (n - a)
    n0 = (n - (k ** 2).sum() / n) / (a - 1)
    return (msb - msw) / (msb + (n0 - 1) * msw)

icc_doi = icc1(d["eg"].values, d["doi_key"].values)
rng = np.random.default_rng(SEED)
dois = d["doi_key"].unique()
boot = []
for _ in range(N_BOOT):
    pick = rng.choice(dois, len(dois), replace=True)
    parts = [d[d["doi_key"] == p].assign(g=f"{p}#{i}") for i, p in enumerate(pick)]
    b = pd.concat(parts)
    boot.append(icc1(b["eg"].values, b["g"].values))
print(f"\nICC(1) of Eg by DOI: {icc_doi:.3f}  [95% CI {np.percentile(boot, 2.5):.3f}, "
      f"{np.percentile(boot, 97.5):.3f}]  (share of Eg variance shared within a paper)")

# ---------------------------------------------------------------------------
# 4. Copied-value check
# ---------------------------------------------------------------------------
hs = d.groupby(["doi_key", "base_compound"])["eg"].agg(["size", "nunique"])
hs = hs[hs["size"] >= 2]
print(f"\nHosts with ≥2 samples in one paper: {len(hs)} | all samples identical Eg: "
      f"{int((hs['nunique'] == 1).sum())} ({(hs['nunique'] == 1).mean():.0%}) — "
      f"check whether host Eg was copied to variants")

# ---------------------------------------------------------------------------
# 5. DOI-grouped CV with composition purging
# ---------------------------------------------------------------------------
META = {"source_row_id", "base_compound", "family_proposed", "formula_base_paper",
        "A_site_paper", "B_site_paper", "doi", "doi_key", "y_log10_1p_final",
        "y_log10_1p", "h2_rate_umol_g_h", "h2_umol_h_rate_mass"}
LEAK = [c for c in X_mat_raw.columns if re.match(r"^(ECB|EVB)_", c)]
feats = [c for c in X_mat_raw.columns if c not in META and c not in LEAK
         and pd.api.types.is_numeric_dtype(X_mat_raw[c])
         and X_mat_raw[c].isna().mean() < 0.50]
d = d.merge(X_mat_raw[["source_row_id"] + feats], on="source_row_id", how="left")
X, y = d[feats].values.astype(float), d["eg"].values
print(f"\nFeatures: {len(feats)} (excluded target-derived: {LEAK})")

def models():
    return {
        "RF": RandomForestRegressor(n_estimators=500, min_samples_leaf=2, max_features=0.3,
                                    n_jobs=-1, random_state=SEED),
        "XGBoost": xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=4,
                                    subsample=0.8, colsample_bytree=0.6, min_child_weight=2,
                                    tree_method="hist", random_state=SEED, verbosity=0),
    }

rows, oof0, purged_frac = [], {}, []
for rep_i in range(N_REPEATS):
    perm = np.random.default_rng(SEED + rep_i).permutation(np.array(sorted(dois)))
    f_of = {g: i % N_FOLDS for i, g in enumerate(perm)}
    f = d["doi_key"].map(f_of).values
    for purge in [False, True]:
        preds = {n: np.full(len(d), np.nan) for n in ["mean", "RF", "XGBoost"]}
        for k in range(N_FOLDS):
            va = np.where(f == k)[0]
            tr = np.where(f != k)[0]
            if purge:
                va_comp = set(d["base_compound"].values[va])
                keep = ~np.isin(d["base_compound"].values[tr], list(va_comp))
                purged_frac.append(1 - keep.mean())
                tr = tr[keep]
            imp = SimpleImputer(strategy="median")
            Xtr, Xva = imp.fit_transform(X[tr]), imp.transform(X[va])
            preds["mean"][va] = y[tr].mean()
            for n, m in models().items():
                m.fit(Xtr, y[tr])
                preds[n][va] = m.predict(Xva)
        for n, p in preds.items():
            rows.append({"repeat": rep_i, "purged": purge, "model": n,
                         "r2": r2_score(y, p), "mae_eV": mean_absolute_error(y, p)})
            if rep_i == 0:
                oof0[(purge, n)] = p

res = pd.DataFrame(rows)
summ = res.groupby(["purged", "model"]).agg(r2=("r2", "mean"), r2_min=("r2", "min"),
                                            r2_max=("r2", "max"),
                                            mae_eV=("mae_eV", "mean")).reset_index()
# cluster bootstrap CI (by DOI) on repeat 0
by = {g: np.where(d["doi_key"].values == g)[0] for g in dois}
ci = []
rng = np.random.default_rng(SEED + 5)
for (purge, n), p in oof0.items():
    vals = []
    for _ in range(N_BOOT):
        idx = np.concatenate([by[g] for g in rng.choice(dois, len(dois), replace=True)])
        vals.append(r2_score(y[idx], p[idx]))
    ci.append({"purged": purge, "model": n, "ci_lo": np.percentile(vals, 2.5),
               "ci_hi": np.percentile(vals, 97.5)})
summ = summ.merge(pd.DataFrame(ci), on=["purged", "model"])
print(f"\n--- DOI-grouped CV, with and without composition purging "
      f"(training rows purged per fold: median {np.median(purged_frac):.0%}) ---")
print(summ.round(3).to_string(index=False))

# ---------------------------------------------------------------------------
# 6. Plot + write
# ---------------------------------------------------------------------------
if len(rep):
    top = rep.head(15).index
    fig, ax = plt.subplots(figsize=(8, 0.4 * len(top) + 1.5))
    for i, comp in enumerate(top):
        v = multi[multi["base_compound"] == comp]["eg"].values
        ax.scatter(v, [i] * len(v), s=25)
        ax.plot([v.min(), v.max()], [i, i], color="grey", lw=0.8)
    ax.set_yticks(range(len(top))); ax.set_yticklabels(top, fontsize=8)
    ax.set_xlabel("Reported Eg (eV), one point per paper")
    ax.set_title("Same composition, different papers")
    plt.tight_layout(); plt.savefig("/tmp/nb15b_reproducibility.png", dpi=130); plt.close()
    print("\nSaved: /tmp/nb15b_reproducibility.png")

spark.createDataFrame(res).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.bandgap_purged_nb15b")
print(f"✓ Written: {SCHEMA}.bandgap_purged_nb15b ({len(res)} rows)")
print("\n=== NB15b COMPLETE ===")